In [ ]:
import os, json
from collections import Counter

train_dir = "/kaggle/input/datasets/hunthi/cnamuangtoun-data/cnamuangtoun/train"
test_dir  = "/kaggle/input/datasets/hunthi/cnamuangtoun-data/cnamuangtoun/test"

LABELS = {"No Fit", "Potential Fit", "Good Fit"}

def get_label(data):
    # Tên key chứa label bị lỗi hiển thị -> tìm theo giá trị cho chắc
    for v in data.values():
        if isinstance(v, str) and v in LABELS:
            return v
    return None

def load_labels(folder):
    labels = []
    for fname in os.listdir(folder):
        if fname.endswith(".json"):
            with open(os.path.join(folder, fname), "r", encoding="utf-8") as f:
                data = json.load(f)
            lbl = get_label(data)
            if lbl:
                labels.append(lbl)
    return labels

train_labels = load_labels(train_dir)
test_labels  = load_labels(test_dir)

print("=== TRƯỚC khi gộp ===")
print(f"Train ({len(train_labels)} samples):", dict(Counter(train_labels)))
print(f"Test  ({len(test_labels)} samples):", dict(Counter(test_labels)))

# Gộp Potential Fit + Good Fit -> Fit
merge_map = {"No Fit": "No Fit", "Potential Fit": "Fit", "Good Fit": "Fit"}
train_labels_merged = [merge_map[l] for l in train_labels]
test_labels_merged  = [merge_map[l] for l in test_labels]

print("\n=== SAU khi gộp ===")
print(f"Train ({len(train_labels_merged)} samples):", dict(Counter(train_labels_merged)))
print(f"Test  ({len(test_labels_merged)} samples):", dict(Counter(test_labels_merged)))

In [ ]:
!pip uninstall -y hf_transfer

In [ ]:
import os, json
from collections import Counter
from sklearn.model_selection import train_test_split

train_dir = "/kaggle/input/datasets/hunthi/cnamuangtoun-data/cnamuangtoun/train"
test_dir  = "/kaggle/input/datasets/hunthi/cnamuangtoun-data/cnamuangtoun/test"

LABELS = {"No Fit", "Potential Fit", "Good Fit"}
merge_map = {"No Fit": "No Fit", "Potential Fit": "Fit", "Good Fit": "Fit"}

def get_label(data):
    for v in data.values():
        if isinstance(v, str) and v in LABELS:
            return v
    return None

def load_files_and_labels(folder):
    files, labels = [], []
    for fname in os.listdir(folder):
        if fname.endswith(".json"):
            with open(os.path.join(folder, fname), "r", encoding="utf-8") as f:
                data = json.load(f)
            lbl = get_label(data)
            if lbl:
                files.append(fname)
                labels.append(merge_map[lbl])
    return files, labels

train_files, train_labels = load_files_and_labels(train_dir)
test_files, test_labels   = load_files_and_labels(test_dir)

train_files_split, val_files_split, train_labels_split, val_labels_split = train_test_split(
    train_files,
    train_labels,
    test_size=0.10,
    random_state=42,
    stratify=train_labels
)

print(f"Train ({len(train_labels_split)} samples):", dict(Counter(train_labels_split)))
print(f"Val   ({len(val_labels_split)} samples):", dict(Counter(val_labels_split)))
print(f"Test  ({len(test_labels)} samples):", dict(Counter(test_labels)))

In [ ]:
!pip install -q transformers sentencepiece einops

import os, json, gc, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             accuracy_score, precision_score, recall_score, f1_score)
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
import warnings; warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
WORK      = "/kaggle/working"
CACHE_DIR = os.path.join(WORK, "cache")
CKPT_DIR  = os.path.join(WORK, "checkpoints")
PLOT_DIR  = os.path.join(WORK, "plots")
RESULTS_CSV = os.path.join(WORK, "results_crossencoder_penalty.csv")
for d in (CACHE_DIR, CKPT_DIR, PLOT_DIR):
    os.makedirs(d, exist_ok=True)

CV_VIEWS = ['skills', 'education', 'year_experience', 'experience', 'responsibilities',
            'projects', 'publications', 'achievement', 'certification', 'employment_history']
JD_VIEWS = ['skills', 'education', 'year_experience_requirement', 'experience_requirement', 'responsibilities']
NUM_PAIRS = len(CV_VIEWS) * len(JD_VIEWS)  # 50

ENCODERS = [
    "BAAI/bge-reranker-v2-m3",
    "jinaai/jina-reranker-v2-base-multilingual",
    "xlm-roberta-base",
    "bert-base-multilingual-cased",
]

MAX_LEN = 256
EPOCHS  = 150
BATCH   = 32

SPLITS = {
    "train": (train_dir, train_files_split, train_labels_split),
    "val":   (train_dir, val_files_split,   val_labels_split),
    "test":  (test_dir,  test_files,        test_labels),
}

In [ ]:
def process_view(view_data):
    """missing_section -> [EMPTY_SECTION]; tra ve (text, exists_flag)."""
    if view_data is None:
        return "[EMPTY_SECTION]", 0
    if isinstance(view_data, list):
        clean = [str(x) for x in view_data if str(x).lower() != "missing_section"]
        if not clean:
            return "[EMPTY_SECTION]", 0
        return ". ".join(clean), 1
    val = str(view_data).lower()
    if val in ("missing_section", "", "none"):
        return "[EMPTY_SECTION]", 0
    return str(view_data), 1


def load_json(path):
    with open(path, "r", encoding="utf-8-sig") as f:
        return json.load(f)


def label_to_bin(label):
    """Dataset moi chi co nhan phan loai (khong co score lien tuc) -> Fit=1, No Fit=0."""
    return 1.0 if label == "Fit" else 0.0


def compute_metrics(y_reg, y_bin, preds):
    preds = np.asarray(preds); y_reg = np.asarray(y_reg); y_bin = np.asarray(y_bin)
    pred_bin = (preds >= 0.5).astype(int)
    return {
        "MAE": mean_absolute_error(y_reg, preds),
        "MSE": mean_squared_error(y_reg, preds),
        "Accuracy": accuracy_score(y_bin, pred_bin),
        "Precision": precision_score(y_bin, pred_bin, zero_division=0),
        "Recall": recall_score(y_bin, pred_bin, zero_division=0),
        "F1-Score": f1_score(y_bin, pred_bin, zero_division=0),
    }


RESULT_COLS = ["Encoding Model", "Fusion", "MAE", "MSE",
               "Accuracy", "Precision", "Recall", "F1-Score"]


def append_result_csv(path, encoder, fusion, m):
    row = {"Encoding Model": encoder, "Fusion": fusion,
           "MAE": round(m["MAE"], 4), "MSE": round(m["MSE"], 4),
           "Accuracy": round(m["Accuracy"], 4), "Precision": round(m["Precision"], 4),
           "Recall": round(m["Recall"], 4), "F1-Score": round(m["F1-Score"], 4)}
    df = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame(columns=RESULT_COLS)
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df = df.drop_duplicates(subset=["Encoding Model", "Fusion"], keep="last")
    df = df[RESULT_COLS]
    df.to_csv(path, index=False, encoding="utf-8-sig")
    return df


def plot_history(hist, title, save_path):
    x = np.arange(1, len(hist["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(x, hist["train_loss"], label=f"Train Loss (final={hist['train_loss'][-1]:.4f})")
    ax1.plot(x, hist["val_loss"],   label=f"Val Loss (final={hist['val_loss'][-1]:.4f})")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
    ax1.set_title(f"{title} - Loss"); ax1.legend(); ax1.grid(True)
    ax2.plot(x, hist["train_acc"], label=f"Train Acc (final={hist['train_acc'][-1]:.4f})")
    ax2.plot(x, hist["val_acc"],   label=f"Val Acc (final={hist['val_acc'][-1]:.4f})")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
    ax2.set_title(f"{title} - Accuracy"); ax2.legend(); ax2.grid(True)
    plt.tight_layout(); plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show(); plt.close()

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download

LOCAL_SAFETENSORS = {
    "bert-base-multilingual-cased": "/kaggle/input/datasets/hunthi/bert-base-multilingual-cased/bert-base-multilingual-cased/model.safetensors",
}

def prepare_local_model_dir(name, cache_root="/kaggle/working/local_models"):
    """Tạo thư mục local: symlink model.safetensors có sẵn + tải config/tokenizer (file nhỏ) từ HF."""
    slug = name.split("/")[-1]
    local_dir = os.path.join(cache_root, slug)
    os.makedirs(local_dir, exist_ok=True)

    target_safetensors = os.path.join(local_dir, "model.safetensors")
    if not os.path.exists(target_safetensors):
        src = LOCAL_SAFETENSORS[name]
        try:
            os.symlink(src, target_safetensors)
        except OSError:
            shutil.copy(src, target_safetensors)  

    needed_files = ["config.json", "tokenizer_config.json", "vocab.txt", "tokenizer.json"]
    for fname in needed_files:
        dst = os.path.join(local_dir, fname)
        if os.path.exists(dst):
            continue
        try:
            fpath = hf_hub_download(repo_id=name, filename=fname)
            shutil.copy(fpath, dst)
        except Exception as e:
            print(f"[{slug}] khong tai duoc {fname} (co the model khong co file nay): {e}")

    return local_dir


# =============================== LOAD ENCODER ===============================
def load_encoder(name):
    slug = name.split("/")[-1]

    load_path = prepare_local_model_dir(name) if name in LOCAL_SAFETENSORS else name

    tok = AutoTokenizer.from_pretrained(load_path, trust_remote_code=True)
    enc = None
    if "jina" in name.lower():
        try:
            enc = AutoModel.from_pretrained(load_path, trust_remote_code=True, torch_dtype=torch.float32)
            print(f"[{slug}] loaded via AutoModel(trust_remote_code)")
        except Exception as e:
            print(f"[{slug}] AutoModel custom that bai -> fallback XLMRobertaModel: {e}")
            from transformers import XLMRobertaModel
            enc, info = XLMRobertaModel.from_pretrained(
                load_path, output_loading_info=True, torch_dtype=torch.float32)
            print(f"[{slug}] missing_keys={len(info['missing_keys'])} "
                  f"unexpected_keys={len(info['unexpected_keys'])} "
                  f"(neu core layer thieu nhieu -> feature jina khong dang tin)")
    else:
        enc = AutoModel.from_pretrained(load_path, trust_remote_code=True)

    enc = enc.to(device).eval()
    for p in enc.parameters():
        p.requires_grad_(False)
    hidden = enc.config.hidden_size
    with torch.no_grad():
        _t = tok(["test"], ["test"], return_tensors="pt", truncation=True, max_length=8).to(device)
        _o = enc(**_t)
    assert hasattr(_o, "last_hidden_state"), \
        f"{slug}: model khong tra ve last_hidden_state (co the nap nham classification head)"
    print(f"[{slug}] hidden_dim={hidden} | last_hidden_state OK")
    return tok, enc, hidden, slug

In [ ]:
# =============== TRICH DAC TRUNG (CROSS-ENCODER: CLS cua cap [cv [SEP] jd]) ===============
@torch.no_grad()
def embed_pairs_cls(cv_texts, jd_texts, tok, enc):
    """50 cap (cv_view_i, jd_view_j) -> CLS (50, hidden)."""
    a = [cv_texts[i] for i in range(len(cv_texts)) for _ in range(len(jd_texts))]
    b = [jd_texts[j] for _ in range(len(cv_texts)) for j in range(len(jd_texts))]
    cls = []
    B = 32
    for i in range(0, len(a), B):
        t = tok(a[i:i+B], b[i:i+B], padding=True, truncation=True,
                max_length=MAX_LEN, return_tensors="pt").to(device)
        o = enc(**t)
        cls.append(o.last_hidden_state[:, 0, :].cpu().numpy().astype(np.float32))
    return np.concatenate(cls, 0)


def prepare_data(split, tok, enc, slug):
    """Moi mau -> CLS (50, hidden) + mask (50). Cache .npz theo slug+split.
    Bo qua an toan file loi/null field thay vi crash ca encoder.
    """
    cache = os.path.join(CACHE_DIR, f"{slug}_{split}_ce.npz")
    if os.path.exists(cache):
        d = np.load(cache)
        print(f"  cache: {cache} | cls={d['cls'].shape}")
        return d["cls"], d["mask"], d["yreg"], d["ybin"]

    folder, files, labels = SPLITS[split]
    all_cls, all_mask, yreg, ybin = [], [], [], []
    n_skipped = 0
    for fname, label in tqdm(list(zip(files, labels)), desc=f"[{slug}] ce {split}"):
        try:
            obj = load_json(os.path.join(folder, fname))
            cv_j = obj.get("resume_structured") or {}  
            jd_j = obj.get("job_description_structured") or {}
            cv_texts, cv_ex = zip(*[process_view(cv_j.get(v)) for v in CV_VIEWS])
            jd_texts, jd_ex = zip(*[process_view(jd_j.get(v)) for v in JD_VIEWS])
            cls = embed_pairs_cls(list(cv_texts), list(jd_texts), tok, enc)
            mask = np.outer(np.array(cv_ex, np.float32), np.array(jd_ex, np.float32)).flatten()
        except Exception as e:
            n_skipped += 1
            print(f"  !! Bo qua file loi {fname}: {e}")
            continue
        all_cls.append(cls); all_mask.append(mask)
        bvar = label_to_bin(label)
        yreg.append(bvar); ybin.append(bvar)

    if n_skipped:
        print(f"  [{slug}|{split}] da bo qua {n_skipped}/{len(files)} file loi")

    cls_arr = np.asarray(all_cls, np.float32)
    mask_arr = np.asarray(all_mask, np.float32)
    yreg = np.asarray(yreg, np.float32); ybin = np.asarray(ybin, np.float32)
    np.savez_compressed(cache, cls=cls_arr, mask=mask_arr, yreg=yreg, ybin=ybin)
    return cls_arr, mask_arr, yreg, ybin

In [ ]:
# =============== DATASET + SCORER + 4 FUSION ===============
class CEDataset(Dataset):
    def __init__(self, cls, mask, yreg, ybin):
        self.cls = torch.tensor(cls, dtype=torch.float32)
        self.mask = torch.tensor(mask, dtype=torch.float32)
        self.yreg = torch.tensor(yreg, dtype=torch.float32).view(-1, 1)
        self.ybin = torch.tensor(ybin, dtype=torch.float32).view(-1, 1)
    def __len__(self): return len(self.cls)
    def __getitem__(self, i):
        return self.cls[i], self.mask[i], self.yreg[i], self.ybin[i]


class ViewPairScorer(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(hidden, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, 1))
        nn.init.constant_(self.fc[3].bias, -2.0)
    def forward(self, cls):
        return torch.sigmoid(self.fc(cls)).squeeze(-1)


class WeightedSumFusion(nn.Module):
    def __init__(self, num_views=50):
        super().__init__(); self.layer = nn.Linear(num_views * 2, 1)
    def forward(self, a_ij, mask):
        return torch.sigmoid(self.layer(torch.cat([a_ij, mask], dim=-1)))


class FusionMLP(nn.Module):
    def __init__(self, num_views=50):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(num_views * 2, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid())
    def forward(self, a_ij, mask):
        return self.net(torch.cat([a_ij, mask], dim=-1))


class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__(); self.ln = nn.LayerNorm(dim); self.fc = nn.Linear(dim, dim)
    def forward(self, x): return x + F.relu(self.fc(self.ln(x)))


class FusionResidualMLP(nn.Module):
    def __init__(self, num_views=50):
        super().__init__()
        self.inp = nn.Sequential(nn.Linear(num_views * 2, 256), nn.ReLU())
        self.res1, self.res2 = ResidualBlock(256), ResidualBlock(256)
        self.hidden = nn.Sequential(nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.2),
                                    nn.Linear(128, 64), nn.ReLU())
        self.out = nn.Sequential(nn.Linear(64, 1), nn.Sigmoid())
    def forward(self, a_ij, mask):
        x = self.inp(torch.cat([a_ij, mask], dim=-1))
        x = self.res1(x); x = self.res2(x); x = self.hidden(x)
        return self.out(x)


class FusionAttention(nn.Module):
    def __init__(self, num_views=50, d_model=64, nhead=8):
        super().__init__()
        self.proj = nn.Linear(2, d_model)  
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                           dim_feedforward=256, dropout=0.2, batch_first=True)
        self.attn = nn.TransformerEncoder(layer, num_layers=3)
        self.head = nn.Sequential(nn.Linear(d_model * num_views, 128), nn.ReLU(),
                                  nn.Dropout(0.2), nn.Linear(128, 1), nn.Sigmoid())
    def forward(self, a_ij, mask):
        b = a_ij.shape[0]
        x = torch.stack([a_ij, mask], dim=-1)   
        h = self.proj(x)
        h = self.attn(h)
        return self.head(h.reshape(b, -1))


class FusionWithScorer(nn.Module):
    def __init__(self, hidden, fusion):
        super().__init__(); self.scorer = ViewPairScorer(hidden); self.fusion = fusion
    def forward(self, cls, mask):
        raw = self.scorer(cls)
        masked = raw * mask
        return self.fusion(masked, mask), masked, raw


FUSION_BUILDERS = {
    "weighted_sum": lambda: WeightedSumFusion(NUM_PAIRS),
    "mlp":          lambda: FusionMLP(NUM_PAIRS),
    "residual_mlp": lambda: FusionResidualMLP(NUM_PAIRS),
    "attention":    lambda: FusionAttention(NUM_PAIRS),
}

In [ ]:
LAMBDA_PENALTY     = 2.0
ALPHA_ACTIVITY     = 0.5
LAMBDA_CONTRASTIVE = 0.3
MARGIN_CONTRASTIVE = 0.5

def contrastive_loss(pred, ybin, margin=MARGIN_CONTRASTIVE):
    pos = ybin       * (1.0 - pred) ** 2
    neg = (1 - ybin) * F.relu(pred - margin) ** 2
    return (pos + neg).mean()

def train_one(model, train_loader, val_loader, ckpt, lr=1e-3):
    model = model.to(device)
    crit_bin = nn.BCELoss()
    opt   = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=10)
    hist  = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best  = float("inf")

    for ep in range(EPOCHS):
        model.train(); tl = 0.0; tp, tt = [], []
        for cls, mask, yr, yb in train_loader:
            cls, mask, yb = cls.to(device), mask.to(device), yb.to(device)
            opt.zero_grad()
            pred, masked, raw = model(cls, mask)

            task          = crit_bin(pred, yb)
            empty_penalty = torch.mean(((1 - mask) * raw) ** 2)
            activity_loss = torch.mean(mask * (0.1 - masked).clamp(min=0))
            contra        = contrastive_loss(pred, yb)

            loss = (task
                    + LAMBDA_PENALTY     * empty_penalty
                    + ALPHA_ACTIVITY     * activity_loss
                    + LAMBDA_CONTRASTIVE * contra)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step(); tl += loss.item()
            tp += list((pred.detach().cpu().numpy().flatten() >= 0.5).astype(int))
            tt += list(yb.cpu().numpy().flatten().astype(int))

        model.eval(); vl = 0.0; vp, vt = [], []
        with torch.no_grad():
            for cls, mask, yr, yb in val_loader:
                cls, mask, yb = cls.to(device), mask.to(device), yb.to(device)
                pred, masked, raw = model(cls, mask)

                v_task  = crit_bin(pred, yb)
                v_pen   = torch.mean(((1 - mask) * raw) ** 2)
                v_act   = torch.mean(mask * (0.1 - masked).clamp(min=0))
                v_contra = contrastive_loss(pred, yb)

                vl += (v_task
                       + LAMBDA_PENALTY     * v_pen
                       + ALPHA_ACTIVITY     * v_act
                       + LAMBDA_CONTRASTIVE * v_contra).item()

                vp += list((pred.cpu().numpy().flatten() >= 0.5).astype(int))
                vt += list(yb.cpu().numpy().flatten().astype(int))

        av = vl / len(val_loader); sched.step(av)
        hist["train_loss"].append(tl / len(train_loader)); hist["val_loss"].append(av)
        hist["train_acc"].append(accuracy_score(tt, tp));  hist["val_acc"].append(accuracy_score(vt, vp))
        if av < best:
            best = av; torch.save(model.state_dict(), ckpt)
    return hist


def evaluate(model, test_loader, ckpt):
    model = model.to(device)
    model.load_state_dict(torch.load(ckpt, map_location=device)); model.eval()
    preds, yr, yb = [], [], []
    with torch.no_grad():
        for cls, mask, r, b in test_loader:
            pred, _, _ = model(cls.to(device), mask.to(device))
            preds += list(pred.cpu().numpy().flatten())
            yr    += list(r.numpy().flatten())
            yb    += list(b.numpy().flatten())
    return compute_metrics(yr, yb, preds)

In [ ]:
def prepare_data(split, tok, enc, slug):
    cache = os.path.join(CACHE_DIR, f"{slug}_{split}_ce.npz")
    if os.path.exists(cache):
        d = np.load(cache)
        print(f"  cache: {cache} | cls={d['cls'].shape}")
        return d["cls"], d["mask"], d["yreg"], d["ybin"]

    folder, files, labels = SPLITS[split]
    all_cls, all_mask, yreg, ybin = [], [], [], []
    n_skipped = 0
    for fname, label in tqdm(list(zip(files, labels)), desc=f"[{slug}] ce {split}"):
        try:
            obj = load_json(os.path.join(folder, fname))
            cv_j = obj.get("resume_structured") or {}
            jd_j = obj.get("job_description_structured") or {}
            cv_texts, cv_ex = zip(*[process_view(cv_j.get(v)) for v in CV_VIEWS])
            jd_texts, jd_ex = zip(*[process_view(jd_j.get(v)) for v in JD_VIEWS])
            cls = embed_pairs_cls(list(cv_texts), list(jd_texts), tok, enc)
            mask = np.outer(np.array(cv_ex, np.float32), np.array(jd_ex, np.float32)).flatten()
        except Exception as e:
            n_skipped += 1
            print(f"  !! Bo qua file loi {fname}: {e}")
            continue
        all_cls.append(cls); all_mask.append(mask)
        bvar = label_to_bin(label)
        yreg.append(bvar); ybin.append(bvar)

    if n_skipped:
        print(f"  [{slug}|{split}] da bo qua {n_skipped}/{len(files)} file loi")

    if len(all_cls) == 0:  
        raise RuntimeError(
            f"[{slug}|{split}] KHONG encode duoc sample nao (tok/enc co the la None hoac loi khac) "
            f"-> khong luu cache de tranh cache rong gay loi ngam ve sau."
        )

    cls_arr = np.asarray(all_cls, np.float32)
    mask_arr = np.asarray(all_mask, np.float32)
    yreg = np.asarray(yreg, np.float32); ybin = np.asarray(ybin, np.float32)
    np.savez_compressed(cache, cls=cls_arr, mask=mask_arr, yreg=yreg, ybin=ybin)
    return cls_arr, mask_arr, yreg, ybin

In [ ]:
from huggingface_hub import login

login("hf_token")

SUFFIX = "_clsonly"
CLS_ONLY_CSV = os.path.join(WORK, "results_classification_only.csv")  

# ENCODERS_TO_RUN = [ENCODERS[0]]      # chi chay bge-reranker-v2-m3
# ENCODERS_TO_RUN = [ENCODERS[1]]      # chi chay jina-reranker-v2
# ENCODERS_TO_RUN = [ENCODERS[2]]      # chi chay xlm-roberta-base
ENCODERS_TO_RUN = [ENCODERS[3]]        # chi chay bert-base-multilingual-cased
# ENCODERS_TO_RUN = ENCODERS           # chay het ca 4

LR_OVERRIDE = {"attention": 1e-3}   

for ENCODER_NAME in ENCODERS_TO_RUN:
    print("\n" + "=" * 80)
    print("ENCODER:", ENCODER_NAME)
    print("=" * 80)
    try:
        tok, enc, hidden, slug = load_encoder(ENCODER_NAME)  

        cls_tr, m_tr, y_tr, b_tr = prepare_data("train", tok, enc, slug)
        cls_v,  m_v,  y_v,  b_v  = prepare_data("val",   tok, enc, slug)
        cls_te, m_te, y_te, b_te = prepare_data("test",  tok, enc, slug)

        del enc, tok; gc.collect(); torch.cuda.empty_cache()

        train_loader = DataLoader(CEDataset(cls_tr, m_tr, y_tr, b_tr), batch_size=BATCH, shuffle=True)
        val_loader   = DataLoader(CEDataset(cls_v,  m_v,  y_v,  b_v),  batch_size=BATCH)
        test_loader  = DataLoader(CEDataset(cls_te, m_te, y_te, b_te), batch_size=BATCH)

        for FUSION_NAME, build in FUSION_BUILDERS.items():
            ckpt = os.path.join(CKPT_DIR, f"{FUSION_NAME}_{slug}{SUFFIX}.pth")   
            model = FusionWithScorer(hidden, build())
            lr = LR_OVERRIDE.get(FUSION_NAME, 1e-3)
            hist = train_one(model, train_loader, val_loader, ckpt, lr=lr)  
            plot_history(hist, f"{FUSION_NAME}{SUFFIX} | {slug}",
                         os.path.join(PLOT_DIR, f"{FUSION_NAME}{SUFFIX}-{slug}.png"))
            m = evaluate(model, test_loader, ckpt)
            df = append_result_csv(CLS_ONLY_CSV, ENCODER_NAME, FUSION_NAME, m)
            print(f"  [{slug} | {FUSION_NAME} | classification-only] "
                  f"Acc={m['Accuracy']:.4f} Precision={m['Precision']:.4f} "
                  f"Recall={m['Recall']:.4f} F1={m['F1-Score']:.4f}")

        del cls_tr, cls_v, cls_te; gc.collect()
    except Exception as e:
        import traceback
        print(f"!! BO QUA encoder {ENCODER_NAME} do loi: {e}")
        traceback.print_exc()
        gc.collect(); torch.cuda.empty_cache()
        continue

print("\nXONG. Bang ket qua classification-only:")
res = pd.read_csv(CLS_ONLY_CSV) if os.path.exists(CLS_ONLY_CSV) else pd.DataFrame()
print("Luu tai:", CLS_ONLY_CSV)
res